# 🔌 大模型工具调用之 MCP (Model Context Protocol) 实战

### 💡 什么是 MCP (Model Context Protocol)？
在前面的实验中，大模型与本地函数的绑定是**紧耦合**的：你需要为每个项目重写一遍 Python 工具函数和 JSON Schema 描述。一旦工具更新，所有调用项目的代码都要改动。

为此，Anthropic 推出了 **MCP (Model Context Protocol, 模型上下文协议)** 行业规范。它的核心目标是：**将大模型客户端与具体的工具服务解耦**。客户端与服务端之间通过标准输入输出 (Stdio) 或网络 (SSE) 采用 JSON-RPC 消息报文实现完全标准的协议交互。

本实验中，我们将引入官方维护的**官方 Filesystem MCP 服务端**，然后在 Notebook 中实现一个**真实的 MCP 客户端**，利用 Stdio 管道连接它们，并由大模型端到端调用 MCP 注册的工具完成工作区目录文件检索与读取任务。**本实验不使用任何 Mock 替代！**

## 📦 依赖库安装提示

为了使用官方的 Model Context Protocol SDK，请确保已经运行过 `pip install mcp` 来安装官方包。我们也可以在下方单元格自动检查并安装。

In [ ]:
!pip show mcp || pip install mcp

## 🛠️ 环境初始化：API 密钥与 OpenAI 客户端配置

为了运行本 Notebook，我们需要配置大模型 API。请在下方填入您的 API Key 和 Base URL。本代码默认支持通义千问 (DashScope) 与硅基流动 (SiliconFlow)，也可以直接使用 OpenAI 或其他兼容的 API 接口。

In [ ]:
import os
import json
import time
import httpx
from openai import OpenAI

# ============================================================
# 👇 请在下方引号内填入您的 API Key（也可以直接读取系统环境变量）
# ============================================================
API_KEY = ""         # 例如: "sk-abc123..."
BASE_URL = "https://api.siliconflow.cn/v1"        # 例如: "https://api.siliconflow.cn/v1" 或 "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "deepseek-ai/DeepSeek-V3"      # 例如: "deepseek-ai/DeepSeek-V3" 或 "qwen-plus"

# 优先从上面填写的变量读取，其次读取系统环境变量
api_key = API_KEY or os.environ.get("OPENAI_API_KEY") or os.environ.get("SILICONFLOW_API_KEY") or os.environ.get("DASH_SCOPE_API_KEY")
base_url = BASE_URL or os.environ.get("OPENAI_API_BASE")
model_name = MODEL_NAME or os.environ.get("OPENAI_API_MODEL")

# 自动识别常见服务商的环境变量
if not base_url:
    if os.environ.get("SILICONFLOW_API_KEY"):
        base_url = "https://api.siliconflow.cn/v1"
        model_name = model_name or "deepseek-ai/DeepSeek-V3"
    elif os.environ.get("DASH_SCOPE_API_KEY"):
        base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
        model_name = model_name or "qwen-plus"
    else:
        base_url = "https://api.openai.com/v1"
        model_name = model_name or "gpt-4o-mini"

if not api_key:
    raise ValueError("❌ 未检测到 API Key！请在上方单元格中配置您的 API Key 或设置系统环境变量。")

# 💡 如果您在 Windows 环境下使用 VPN 或代理工具，可能会遇到 SSL/ConnectError 报错。
# 此时可以尝试取消下方这行代码的注释，使用禁用了 SSL 证书验证的自定义 httpx 客户端：
# openai_client = OpenAI(api_key=api_key, base_url=base_url, http_client=httpx.Client(verify=False))
openai_client = OpenAI(api_key=api_key, base_url=base_url)
print(f"✨ 客户端实例化成功！当前使用的接口地址为: {base_url}，模型为: {model_name}")

## 📂 测试数据生成

为了进行文件查找实验，我们需要在本地创建一个测试目录 `data/fc_test`，并在其中放入几个测试文件，包括我们要让大模型查找的目标文件 `target_simple.txt`。

In [ ]:
import shutil

# 创建测试目录
test_dir = "data/fc_test"
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)
os.makedirs(test_dir, exist_ok=True)

# 写入一些干扰文件与目标文件
files_to_create = {
    "report.txt": "2026年年度财务报告：公司业绩稳步上升。",
    "config.json": '{"port": 8080, "debug": false}',
    "target_simple.txt": "恭喜你！成功找到了 target_simple.txt 文件。密钥为: SIMPLE_SUCCESS_2026",
    "notes.log": "2026-06-03 12:00:00 - Server started successfully."
}

for filename, content in files_to_create.items():
    with open(os.path.join(test_dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"💾 测试目录 {test_dir} 准备完毕，已创建 {len(files_to_create)} 个文件。")

---
## 🧱 第一步：配置官方 Filesystem MCP 服务端

在真实应用场景中，我们很少从零编写所有的基础工具（如文件操作、数据库查询、网页检索等），而是直接使用社区或官方成熟的现成 MCP 服务端。
Anthropic 官方提供了一个名为 `@modelcontextprotocol/server-filesystem` 的现成文件系统 MCP 服务端。我们可以通过 `npx` (Windows 环境下使用 `npx.cmd`) 极其方便地免安装拉起它，并向其传入我们允许访问的绝对目录路径（例如 `data/fc_test`）。

In [ ]:
import os

# 我们这里定义一个变量来存储要操作的文件目录的绝对路径，作为 MCP 服务器的安全白名单根目录。
allowed_dir = os.path.abspath("data/fc_test")
print(f"✅ 允许 MCP 服务器访问的绝对目录路径为: {allowed_dir}")

## 🔌 第二步：构建真实的 MCP 客户端 stdio 通信环境

在真实世界中，大模型前端（如 Claude Desktop 或 IDE 助手）会将服务器程序作为一个后台子进程运行，并通过管道（Stdio）进行通信。

由于 MCP SDK 是基于 `asyncio` 的，而在 Jupyter Notebook 的主线程中已经运行着一个 Jupyter 的事件循环，为了不干扰 Jupyter 环境并且能在单元格中顺序得到执行结果，我们编写一个简单的多线程线程隔离异步执行器 `run_async`，确保代码 100% 稳妥运行。

In [ ]:
import threading
import asyncio

def run_async(coro):
    """
    在新线程中运行一个独立的 asyncio 事件循环，确保在 Jupyter Notebook 中不会因为 event loop 已在运行而抛出异常。
    """
    result = []
    error = []
    
    def target():
        try:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            res = loop.run_until_complete(coro)
            result.append(res)
        except Exception as e:
            error.append(e)
        finally:
            loop.close()
            
    thread = threading.Thread(target=target)
    thread.start()
    thread.join()
    
    if error:
        raise error[0]
    return result[0]

print("✅ 线程隔离异步执行器 run_async 编译成功。")

## 🧪 第三步：通过 MCP Client 发现服务并进行工具调用测试

我们将使用 `StdioServerParameters` 启动官方的 `@modelcontextprotocol/server-filesystem` 服务端。在客户端，我们将拉取服务端暴露的所有文件操作工具，并手动触发调用 `read_text_file` 读取目标文件，观察其标准交互流程。

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import sys
import os

async def test_mcp_connection():
    # 配置 MCP 启动参数：Windows 下使用 npx.cmd，类 Unix 系统使用 npx
    command = "npx.cmd" if os.name == 'nt' else "npx"
    allowed_dir = os.path.abspath("data/fc_test")
    
    server_params = StdioServerParameters(
        command=command,
        args=["-y", "@modelcontextprotocol/server-filesystem", allowed_dir],
        env=None
    )
    
    # 启动双端 stdio 管道连接
    async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
        async with ClientSession(read, write) as session:
            # 初始化连接
            await session.initialize()
            
            # 1. 客户端自动拉取服务端暴露的工具列表
            tools_response = await session.list_tools()
            print("🔌 [MCP 客户端发现的工具列表]：")
            for tool in tools_response.tools:
                print(f"   - 工具名称: {tool.name}")
                print(f"     ├─ 描述: {tool.description[:60]}...")
                print(f"     └─ 参数 Schema: {tool.inputSchema}")
            
            # 2. 客户端直接发起工具调用测试
            target_file_path = os.path.join(allowed_dir, "target_simple.txt")
            print(f"\n⚙️ 发起工具调用: read_text_file(path='{target_file_path}')...")
            result = await session.call_tool("read_text_file", arguments={"path": target_file_path})
            print(f"📥 [MCP 服务端响应内容]：\n{result.content[0].text}")

# 运行测试
run_async(test_mcp_connection())

## 🤖 第四步：大模型结合 MCP 客户端实现端到端 Agent 检索

为了实现解耦，大模型客户端并不直接硬编码工具。我们会在启动 MCP 会话后：
1. 从 MCP 服务器获取支持的工具；
2. **自动将其转换为大模型（OpenAI SDK）认可的 tool_schema**；
3. 呼叫大模型，若大模型决定调用工具，则将调用请求转发给 MCP 服务端；
4. 将 MCP 的执行结果反馈给大模型，获取最终答案。

In [ ]:
def convert_mcp_to_openai_tools(mcp_tools) -> list:
    """
    辅助函数：将 MCP 暴露的工具 Schema 转换为 OpenAI Tool Schema
    """
    openai_tools = []
    for tool in mcp_tools.tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.inputSchema
            }
        })
    return openai_tools

async def run_mcp_agent_loop():
    command = "npx.cmd" if os.name == 'nt' else "npx"
    allowed_dir = os.path.abspath("data/fc_test")
    
    server_params = StdioServerParameters(
        command=command,
        args=["-y", "@modelcontextprotocol/server-filesystem", allowed_dir],
        env=None
    )
    
    # 给大模型提示明确可访问的绝对目录，并让其在该目录中找出并读取目标文件
    user_query = f"请帮我查一下在目录 '{allowed_dir}' 下面，有一个名为 target_simple.txt 的文件。请帮我找出并把它的内容告诉我。"
    messages = [{"role": "user", "content": user_query}]
    
    async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
        async with ClientSession(read, write) as session:
            # 初始化 MCP
            await session.initialize()
            
            # 1. 自动读取并转换工具
            mcp_tools = await session.list_tools()
            openai_tools = convert_mcp_to_openai_tools(mcp_tools)
            
            # 2. 发送给大模型进行首次决策
            print("📤 呼叫大模型（附带 MCP 转换好的工具集）...")
            response = openai_client.chat.completions.create(
                model=model_name,
                messages=messages,
                tools=openai_tools,
                tool_choice="auto"
            )
            
            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls
            messages.append(assistant_message)
            
            if tool_calls:
                print("🤖 大模型做出调用工具决策：")
                for call in tool_calls:
                    func_name = call.function.name
                    func_args = json.loads(call.function.arguments)
                    print(f"   - 申请调用: {func_name} | 参数: {func_args}")
                    
                    # 3. 将调用转发给 MCP 服务端执行
                    mcp_response = await session.call_tool(func_name, arguments=func_args)
                    mcp_output = mcp_response.content[0].text
                    print(f"   ▶️ MCP 服务端物理执行反馈: '{mcp_output}'")
                    
                    # 反馈给大模型
                    messages.append({
                        "role": "tool",
                        "tool_call_id": call.id,
                        "name": func_name,
                        "content": mcp_output
                    })
                
                # 4. 获取最终答案
                print("\n📤 反馈结果给大模型，正在生成最终回答...")
                final_response = openai_client.chat.completions.create(
                    model=model_name,
                    messages=messages
                )
                print("\n🤖 大模型结合 MCP 结果的最终答复:")
                print("=" * 65)
                print(final_response.choices[0].message.content)
                print("=" * 65)
            else:
                print("大模型无需使用工具，答复为:")
                print(assistant_message.content)

# 运行端到端 Agent
run_async(run_mcp_agent_loop())

## 📈 课后知识复盘与思考练习

### 🧠 思考题
1. MCP 的核心意义在于“客户端与工具服务器解耦”。请设想：如果在您的学校或企业内，有很多个不同的 LLM 项目（如自动批改作业系统、选课助手、科研检索助手）都需要访问学生数据库，如果不用 MCP 应当怎么做？如果使用 MCP，系统的维护成本和安全性会发生怎样的改善？